In [10]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from scipy.integrate import solve_ivp
from IPython.display import HTML, Image

In [15]:
class SpringPendulum:
        #Represents a physical spring pendulum system using lagrangian mechanics
    def __init__(self, mass = 1.0, length = 1.0, gravity = 9.81, spring_const = 50.0):
        self.m = float(mass)
        self.l = float(length)
        self.g = float(gravity)
        self.k = float(spring_const)

        #simulation containers
        self.time_steps = None
        self.state_history = None #stores the raw data from [x, theta, v_x, omega]

    def equations_of_motion(self, t, state):
        x, theta, v_x, omega = state

        current_length = self.l + x
        if current_length <= 0.01:
            current_length = 0.01 #prevents division by zero error

        #First-order equations derived via lagrangian mechanics
        dxdt = v_x
        dthetadt = omega
        dv_xdt = current_length*(omega**2)+self.g*np.cos(theta)-(self.k/self.m)*x
        domegadt = (-2*v_x*omega/current_length)-(self.g*np.sin(theta)/current_length)

        return [dxdt, dthetadt, dv_xdt, domegadt]

    def simulate(self, initial_state, duration = 20.0, points = 500):
        self.time_steps = np.linspace(0, duration, points)

        solution = solve_ivp(
            fun = self.equations_of_motion, 
            t_span = (0, duration), 
            y0 = initial_state, 
            t_eval = self.time_steps, 
            method = 'RK45', 
        )

        self.state_history = solution.y
        
        return self.state_history

    def get_cartesian_coordinates(self):
        if self.state_history is None:
            raise ValueError("No simulation history found. Run .simulate() first.")

        x_stretch = self.state_history[0]
        theta_angle = self.state_history[1]

        x_cartesian = (self.l + x_stretch)*np.sin(theta_angle)
        y_cartesian = -(self.l + x_stretch)*np.cos(theta_angle)

        return x_cartesian, y_cartesian

pendulum = SpringPendulum(mass = 1.0, length = 1.0, gravity = 9.81, spring_const = 40.0)
        
#Set initial state [x_stretch, angle (theta), v_x, omega]
initial_state = [0.2, np.radians(45),0.0,0.0] #Starts with a 0.2m stretch and released at a 45 degree angle
    
#Running the simulation
duration = 20.0
points = 400
pendulum.simulate(initial_state, duration=duration, points=points)
x_data, y_data = pendulum.get_cartesian_coordinates()

#Plot
fig, ax = plt.subplots()
max_reach = pendulum.l + max(abs(pendulum.state_history[0])) + 0.5
ax.set_xlim(-max_reach,max_reach)
ax.set_ylim(-max_reach,0.5)
ax.set_aspect('equal')
ax.grid(True)
ax.set_title('Spring Pendulum Simulation')

#animation elements
trace, = ax.plot([],[], '-',lw=1, color='red', label='path trace', alpha=0.5)
line, = ax.plot([],[], 'o-', lw=2, color='blue', label='Spring')
bob, = ax.plot([],[], 'o', markersize=12, color='black',label='Mass')
ax.legend()

def init():
    trace.set_data([],[])
    line.set_data([],[])
    bob.set_data([],[])
    return trace, line, bob

def update(frame):
    trace.set_data(x_data[:frame],y_data[:frame])
    line.set_data([0,x_data[frame]],[0,y_data[frame]])
    bob.set_data([x_data[frame]],[y_data[frame]])
    return trace, line, bob

calc_interval=(1000*duration)/points

ani = animation.FuncAnimation(
    fig,
    update,
    frames=points,
    init_func=init,
    blit=True,
    interval=calc_interval
)

plt.close()

ani.save('Spring_Pendulum.gif', writer = 'pillow', fps=15)
Image(url='Spring_Pendulum.gif')